# Remote MLIP interface workflows

Generate and optionally submit the four validated GPU workflows: ORB (Si/Ge), DPA (LiCoO₂), MatRIS (MgO/SrTiO₃), and SevenNet with bounded MLIP gradient descent (ZnO/GaN).

Prerequisites: configured `~/.jfremote/std.yaml`, workers named `orb`, `dpa`, `matris`, and `sevenn`, plus compatible checkpoints.

In [ ]:
from copy import deepcopy
from pathlib import Path
import json
import sys

import InterOptimus

candidates = (Path.cwd(), Path.cwd().parent, Path(InterOptimus.__file__).resolve().parents[1])
REPO_ROOT = next((path for path in candidates if (path / "scripts").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the InterOptimus repository or its examples directory")
sys.path.insert(0, str(REPO_ROOT))

from InterOptimus.agents.simple_iomaker import run_simple_iomaker
from scripts.prepare_interface_workflow_validation import matrix, structures

WORKDIR = Path.cwd() / "remote_mlip_examples"
SUBMIT = False  # Set True only after checking workers, checkpoints, and partitions.
WORKDIR.mkdir(exist_ok=True)
(WORKDIR / "materials").mkdir(exist_ok=True)
(WORKDIR / "configs").mkdir(exist_ok=True)

materials = structures()
for name, structure in materials.items():
    structure.to(filename=WORKDIR / "materials" / f"{name}.cif")

wanted = {
    "validate_mlip_orb_si_ge",
    "validate_mlip_dpa_lco",
    "validate_mlip_matris_mgo_sto",
    "validate_mlip_sevenn_zno_gan",
}
configs = []
for source in matrix():
    if source["workflow_name"] not in wanted:
        continue
    config = deepcopy(source)
    bulk = config["IO_workflow_config"]["bulk_cifs"]
    bulk["film_cif"] = str((WORKDIR / bulk["film_cif"]).resolve())
    bulk["substrate_cif"] = str((WORKDIR / bulk["substrate_cif"]).resolve())
    path = WORKDIR / "configs" / f"{config['workflow_name']}.json"
    path.write_text(json.dumps(config, indent=2))
    configs.append((path, config))

[(config["workflow_name"], path) for path, config in configs]

In [ ]:
results = {}
if SUBMIT:
    for _, config in configs:
        result = run_simple_iomaker(config)
        results[config["workflow_name"]] = result
        print(config["workflow_name"], result["mlip_job_uuid"])
else:
    print("Configuration only. Review the JSON files, then set SUBMIT = True.")

results